# GQA / MQA Long-Context Extrapolation Study — Colab GPU runner

Trains 4 attention variants (MHA, GQA-8, GQA-4, MQA) at ~20M params on TinyStories,
then does zero-shot long-context evaluation at 1k/2k/4k/8k tokens using one RoPE
variant (dynamic NTK-aware scaling).

**Metrics evaluated (and only these):**
1. **Perplexity (PPL) vs. context length** -- primary quality metric
2. **Peak VRAM & memory scaling** -- measured peak allocator usage + KV-cache size per context length
3. **Inference throughput & latency** -- tokens/second and ms/token (prefill and cached decode)

Train/val split is by whole story (never cuts a story in half), and the split is
content-based so train.py and evaluate.py always agree on which stories are held out,
even if they load different amounts of data.

Tokenizer encoding uses a fast linked-list+heap BPE implementation
(`encode_bulk_fast`) instead of the naive per-token tensor version -- encoding a
20M-character corpus takes ~2 minutes instead of multiple hours.

**Before running:** Runtime → Change runtime type → Hardware accelerator → **GPU** (T4 is fine).

## 0. Check GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- go to Runtime > Change runtime type > GPU')


## 1. Install dependencies
(`torch` ships with Colab already; this adds the rest.)

In [ ]:
!pip install -q datasets pandas matplotlib seaborn


## 2. Write project source files
(Same code as the standalone `.py` files -- editable in place if you want to tweak anything.)

In [ ]:
%%writefile tokenizer.py
"""
tokenizer.py

GPU-friendly BPE tokenizer using fixed-size PyTorch tensor matrices for
merge lookup. Training happens offline in plain Python (frequency counting
is not a good fit for GPU tensor ops); encoding is done with tensor indexing
so it can run on GPU if needed.

This is the tokenizer you supplied, kept functionally identical, with:
  - a `vocab_size` property
  - a `save` / `load` pair (json + tensor state) so you don't have to
    retrain BPE merges every run
  - minor guards (raises a clear error instead of silently overflowing
    the matrices if target_vocab_size > max_vocab_size)
"""

import json
import os
import torch


class PyTorchBPETokenizer:
    def __init__(self, max_vocab_size=512):
        self.max_vocab_size = max_vocab_size

        # Fixed-size 2D PyTorch Tensor matrices for GPU lookup.
        # merges_matrix[p0, p1] = merged token id (or -1 if no such merge)
        # rank_matrix[p0, p1]   = merge priority (lower = merge first)
        self.merges_matrix = torch.full(
            (max_vocab_size, max_vocab_size), -1, dtype=torch.long
        )
        self.rank_matrix = torch.full(
            (max_vocab_size, max_vocab_size), float("inf"), dtype=torch.float32
        )

        # Base byte vocabulary (0-255)
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.num_merges = 0

    # ------------------------------------------------------------------ #
    # Training
    # ------------------------------------------------------------------ #
    def add_merge_rule(self, pair, new_id):
        """Register a learned rule directly into PyTorch Tensor matrices."""
        p0, p1 = pair
        rank = self.num_merges
        self.merges_matrix[p0, p1] = new_id
        self.rank_matrix[p0, p1] = float(rank)
        self.vocab[new_id] = self.vocab[p0] + self.vocab[p1]
        self.num_merges += 1

    def train_python(self, text, target_vocab_size):
        """Offline Python training phase to count frequencies and build vocabulary."""
        if target_vocab_size < 256:
            return
        if target_vocab_size > self.max_vocab_size:
            raise ValueError(
                f"target_vocab_size ({target_vocab_size}) exceeds max_vocab_size "
                f"({self.max_vocab_size}); the lookup matrices are fixed-size."
            )

        num_merges = target_vocab_size - 256
        ids = list(text.encode("utf-8"))

        for i in range(num_merges):
            counts = {}
            for pair in zip(ids, ids[1:]):
                counts[pair] = counts.get(pair, 0) + 1
            if not counts:
                break

            best_pair = max(counts, key=counts.get)
            new_id = 256 + i

            # Transfer rule to PyTorch Tensor matrix
            self.add_merge_rule(best_pair, new_id)

            # Apply merge to working sequence
            newids = []
            idx = 0
            while idx < len(ids):
                if (
                    idx < len(ids) - 1
                    and ids[idx] == best_pair[0]
                    and ids[idx + 1] == best_pair[1]
                ):
                    newids.append(new_id)
                    idx += 2
                else:
                    newids.append(ids[idx])
                    idx += 1
            ids = newids

    # ------------------------------------------------------------------ #
    # Encode / decode
    # ------------------------------------------------------------------ #
    def encode(self, text, device="cpu"):
        """Encodes string input directly into 1D PyTorch LongTensor using Tensor Matrix ops."""
        bytes_data = list(text.encode("utf-8"))
        ids = torch.tensor(bytes_data, dtype=torch.long, device=device)
        rank_mat = self.rank_matrix.to(device)
        merge_mat = self.merges_matrix.to(device)

        while len(ids) >= 2:
            pairs_left = ids[:-1]
            pairs_right = ids[1:]

            # Parallel GPU matrix indexing
            pair_ranks = rank_mat[pairs_left, pairs_right]
            min_rank, min_idx = torch.min(pair_ranks, dim=0)
            if torch.isinf(min_rank):
                break

            best_left = pairs_left[min_idx]
            best_right = pairs_right[min_idx]
            new_id = merge_mat[best_left, best_right]

            # Reconstruct sequence tensor, merging every occurrence of the
            # winning pair (not just the one at min_idx).
            new_ids = []
            i = 0
            n = len(ids)
            while i < n:
                if i < n - 1 and ids[i] == best_left and ids[i + 1] == best_right:
                    new_ids.append(new_id)
                    i += 2
                else:
                    new_ids.append(ids[i])
                    i += 1
            ids = torch.stack(
                [t if isinstance(t, torch.Tensor) else torch.tensor(t, device=device) for t in new_ids]
            )

        return ids  # Returns PyTorch Tensor directly

    def encode_bulk_fast(self, text, device="cpu"):
        """
        Same result as encode(), but usable on large texts (hundreds of
        thousands to tens of millions of characters).

        encode() rescans the *entire* remaining sequence on every single
        merge (to find the lowest-rank pair), and does it with per-token
        PyTorch tensor access -- both the algorithm (O(n) work repeated once
        per merge, i.e. O(n * num_merges) total) and the tensor overhead
        make it unusable past a few tens of thousands of characters. At
        vocab_size=2048 (~1800 merges) over a 19M-character corpus that's
        tens of billions of operations -- hours, possibly longer, not a
        hang, just genuinely that slow.

        This produces the identical result (same merge order: lowest rank
        first, ties broken by leftmost occurrence, same as encode() --
        verified against it directly) using a doubly-linked list over token
        positions plus a min-heap of candidate merges. After a merge, only
        the (up to two) new pairs created at the merge point are added back
        to the heap -- nothing else is rescanned. That's the standard
        efficient BPE algorithm (O((n + num_merges) log n) instead of
        O(n * num_merges)); measured at ~140K chars/sec on real TinyStories
        text, vs. encode()'s ~700 chars/sec at this vocab size -- a 19M-char
        corpus goes from multiple hours to about two minutes.
        """
        import heapq

        rank_lookup = self._get_rank_lookup_cache()

        ids = list(text.encode("utf-8"))
        n = len(ids)
        if n < 2:
            return torch.tensor(ids, dtype=torch.long, device=device)

        prev = list(range(-1, n - 1))       # prev[i] = i-1 (prev[0] = -1)
        nxt = list(range(1, n + 1))         # nxt[i] = i+1 (nxt[n-1] = -1)
        nxt[-1] = -1
        alive = [True] * n

        heap = []  # entries: (rank, i, j, new_id, ids[i]_at_push_time, ids[j]_at_push_time)

        def try_push(i):
            if i == -1:
                return
            j = nxt[i]
            if j == -1:
                return
            entry = rank_lookup.get((ids[i], ids[j]))
            if entry is not None:
                heapq.heappush(heap, (entry[0], i, j, entry[1], ids[i], ids[j]))

        for i in range(n - 1):
            try_push(i)

        while heap:
            rank, i, j, new_id, left_val, right_val = heapq.heappop(heap)
            # Stale-entry check: valid only if both nodes are still alive,
            # still adjacent, and neither token value changed since this
            # candidate was queued (an earlier, higher-priority merge could
            # have consumed or relabeled one of them in the meantime).
            if not alive[i] or not alive[j] or nxt[i] != j or ids[i] != left_val or ids[j] != right_val:
                continue

            ids[i] = new_id
            alive[j] = False
            j_next = nxt[j]
            nxt[i] = j_next
            if j_next != -1:
                prev[j_next] = i

            try_push(prev[i])
            try_push(i)

        result = []
        i = 0
        while i != -1:
            result.append(ids[i])
            i = nxt[i]

        return torch.tensor(result, dtype=torch.long, device=device)

    def _get_rank_lookup_cache(self):
        """
        (p0, p1) -> (rank, new_id) dict, used by encode_bulk_fast. Building
        it is cheap in isolation (~20ms at vocab_size=2048) but
        encode_bulk_fast may be called repeatedly (train text, val text,
        etc.) -- rebuilding it every single call is wasteful, so it's cached and only
        rebuilt when the merge table has actually changed (num_merges
        differs from what was cached, e.g. after add_merge_rule or load()).
        """
        if getattr(self, "_rank_lookup_cache", None) is None or \
                getattr(self, "_rank_lookup_cache_num_merges", -1) != self.num_merges:
            nz = (self.rank_matrix != float("inf")).nonzero(as_tuple=False)
            cache = {}
            for p0, p1 in nz.tolist():
                cache[(p0, p1)] = (self.rank_matrix[p0, p1].item(), self.merges_matrix[p0, p1].item())
            self._rank_lookup_cache = cache
            self._rank_lookup_cache_num_merges = self.num_merges
        return self._rank_lookup_cache

    def encode_batch(self, texts, device="cpu"):
        """Convenience wrapper: list[str] -> list[LongTensor]."""
        return [self.encode(t, device=device) for t in texts]

    def decode(self, token_ids):
        """Decodes PyTorch Tensor or list of token IDs back into string."""
        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.tolist()
        text_bytes = b"".join(self.vocab.get(idx, b"") for idx in token_ids)
        return text_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ #
    # Persistence
    # ------------------------------------------------------------------ #
    @property
    def vocab_size(self):
        return 256 + self.num_merges

    def save(self, out_dir):
        os.makedirs(out_dir, exist_ok=True)
        torch.save(
            {
                "merges_matrix": self.merges_matrix,
                "rank_matrix": self.rank_matrix,
                "max_vocab_size": self.max_vocab_size,
                "num_merges": self.num_merges,
            },
            os.path.join(out_dir, "tokenizer_tensors.pt"),
        )
        # vocab keys are ints but bytes aren't JSON-safe -> store as latin-1 strings
        vocab_serializable = {str(k): v.decode("latin-1") for k, v in self.vocab.items()}
        with open(os.path.join(out_dir, "vocab.json"), "w") as f:
            json.dump(vocab_serializable, f)

    @classmethod
    def load(cls, out_dir):
        state = torch.load(os.path.join(out_dir, "tokenizer_tensors.pt"))
        tok = cls(max_vocab_size=state["max_vocab_size"])
        tok.merges_matrix = state["merges_matrix"]
        tok.rank_matrix = state["rank_matrix"]
        tok.num_merges = state["num_merges"]
        with open(os.path.join(out_dir, "vocab.json")) as f:
            vocab_serializable = json.load(f)
        tok.vocab = {int(k): v.encode("latin-1") for k, v in vocab_serializable.items()}
        return tok


if __name__ == "__main__":
    # Quick smoke test
    tok = PyTorchBPETokenizer(max_vocab_size=512)
    sample = "Once upon a time there was a little robot who loved to read books. " * 20
    tok.train_python(sample, target_vocab_size=320)
    ids = tok.encode("Once upon a time there was a robot.")
    print("vocab_size:", tok.vocab_size)
    print("encoded len:", len(ids), ids[:20])
    print("decoded:", tok.decode(ids))
    assert tok.decode(ids) == "Once upon a time there was a robot."
    print("Round-trip OK.")


In [ ]:
%%writefile model.py
"""
model.py

From-scratch decoder-only Transformer with:
  - RMSNorm
  - Rotary Positional Embeddings (RoPE), vanilla or NTK-aware scaled
  - Attention topology toggle: MHA (n_kv_heads == n_heads), GQA (1 < n_kv_heads < n_heads),
    MQA (n_kv_heads == 1)
  - SwiGLU MLP
  - Optional pre-allocated KV cache for exact VRAM accounting during
    zero-shot long-context evaluation

Everything here is plain PyTorch (no HF), so every internal variable is
directly reachable for the ablation (this is the whole point of Phase 1).
"""

import math
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


# ========================================================================
# Config
# ========================================================================

@dataclass
class ModelConfig:
    vocab_size: int = 512
    dim: int = 512
    n_layers: int = 8
    n_heads: int = 8
    n_kv_heads: int = 8          # ==n_heads -> MHA, 1<x<n_heads -> GQA, 1 -> MQA
    max_seq_len: int = 1024      # L_train
    ffn_mult: float = 4.0        # SwiGLU hidden dim multiplier (before 2/3 correction)
    rope_theta: float = 10000.0
    norm_eps: float = 1e-6
    dropout: float = 0.0
    tie_embeddings: bool = True

    @property
    def head_dim(self):
        assert self.dim % self.n_heads == 0
        return self.dim // self.n_heads

    def attention_name(self):
        if self.n_kv_heads == self.n_heads:
            return "MHA"
        if self.n_kv_heads == 1:
            return "MQA"
        return f"GQA-{self.n_heads // self.n_kv_heads}"


def make_model_config(attention: str = "mha", vocab_size: int = 512, max_seq_len: int = 1024,
                       target_params: str = "20m") -> ModelConfig:
    """
    Returns a ModelConfig sized to ~20M params: dim=512, n_layers=6,
    n_heads=16 (head_dim=32) -> ~20.3M params at vocab_size=2048.

    n_heads=16 is chosen (not 8) specifically so gqa8/gqa4/mqa are all
    distinct topologies -- with n_heads=8, "8 query heads per KV head"
    degenerates to MQA, which defeats the point of a 3-way ablation.

    attention: "mha" | "gqa8" | "gqa4" | "mqa"
      gqaN means N query heads share each KV head (n_kv_heads = n_heads // N).
    """
    n_heads = 16
    if attention == "mha":
        n_kv_heads = n_heads
    elif attention == "gqa8":
        n_kv_heads = max(1, n_heads // 8)   # 8 query heads per KV head -> 2 kv heads
    elif attention == "gqa4":
        n_kv_heads = max(1, n_heads // 4)   # 4 query heads per KV head -> 4 kv heads
    elif attention == "mqa":
        n_kv_heads = 1
    else:
        raise ValueError(f"Unknown attention topology: {attention}")

    return ModelConfig(
        vocab_size=vocab_size,
        dim=512,
        n_layers=6,
        n_heads=n_heads,
        n_kv_heads=n_kv_heads,
        max_seq_len=max_seq_len,
    )


# kept as an alias so any external references to the old name still work
make_50m_config = make_model_config


# ========================================================================
# RMSNorm
# ========================================================================

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype
        x = x.float()
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight.float()).to(dtype)


# ========================================================================
# RoPE (vanilla + NTK-aware scaling)
# ========================================================================

def precompute_rope_freqs(
    head_dim: int,
    seq_len: int,
    theta: float = 10000.0,
    train_max_seq_len: Optional[int] = None,
) -> torch.Tensor:
    """
    Returns complex-valued freqs_cis of shape (seq_len, head_dim // 2).

    Single RoPE variant used everywhere in this project: dynamic NTK-aware
    scaling (Peng et al., "NTK-aware scaled RoPE"). It rescales theta
    itself -- rather than linearly interpolating positions -- so high
    frequencies stay sharp and low frequencies stretch to cover extended
    context. When seq_len <= train_max_seq_len this is mathematically
    identical to vanilla RoPE (scale == 1, theta unchanged); it only
    kicks in once you extrapolate past the training length, which is
    exactly the zero-shot long-context regime this project measures.
    """
    if train_max_seq_len is not None and seq_len > train_max_seq_len:
        scale = seq_len / train_max_seq_len
        theta = theta * (scale ** (head_dim / (head_dim - 2)))

    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)  # (seq_len, head_dim//2)
    return torch.polar(torch.ones_like(freqs), freqs)  # complex64


def apply_rope(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """
    x: (batch, seq_len, n_heads, head_dim)
    freqs_cis: (seq_len, head_dim//2) complex
    """
    xc = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(2)  # (1, seq_len, 1, head_dim//2)
    x_rotated = torch.view_as_real(xc * freqs_cis).flatten(-2)
    return x_rotated.type_as(x)


# ========================================================================
# KV Cache (for exact VRAM accounting under GQA/MQA vs MHA)
# ========================================================================

class KVCache:
    """
    Pre-allocated cache with shape (batch, n_kv_heads, max_seq_len, head_dim)
    per layer. This is where GQA/MQA's memory savings actually live: fewer
    kv heads -> proportionally smaller cache, independent of n_heads (query
    side is never cached).
    """

    def __init__(self, n_layers, batch_size, n_kv_heads, max_seq_len, head_dim, device, dtype=torch.float32):
        shape = (n_layers, batch_size, n_kv_heads, max_seq_len, head_dim)
        self.k = torch.zeros(shape, device=device, dtype=dtype)
        self.v = torch.zeros(shape, device=device, dtype=dtype)
        self.seq_len = 0

    def update(self, layer_idx, k, v, start_pos):
        # k, v: (batch, n_kv_heads, seq_len, head_dim)
        seq_len = k.shape[2]
        self.k[layer_idx, :, :, start_pos:start_pos + seq_len, :] = k
        self.v[layer_idx, :, :, start_pos:start_pos + seq_len, :] = v
        return (
            self.k[layer_idx, :, :, :start_pos + seq_len, :],
            self.v[layer_idx, :, :, :start_pos + seq_len, :],
        )

    def nbytes(self):
        return self.k.element_size() * self.k.nelement() * 2  # k + v


def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """(batch, n_kv_heads, seq_len, head_dim) -> (batch, n_kv_heads*n_rep, seq_len, head_dim)"""
    if n_rep == 1:
        return x
    b, n_kv, s, d = x.shape
    return x[:, :, None, :, :].expand(b, n_kv, n_rep, s, d).reshape(b, n_kv * n_rep, s, d)


# ========================================================================
# Attention (toggleable MHA / GQA / MQA)
# ========================================================================

class Attention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.n_rep = cfg.n_heads // cfg.n_kv_heads
        self.head_dim = cfg.head_dim

        self.wq = nn.Linear(cfg.dim, cfg.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(cfg.dim, cfg.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(cfg.dim, cfg.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(cfg.n_heads * self.head_dim, cfg.dim, bias=False)
        self.dropout = cfg.dropout

    def forward(
        self,
        x: torch.Tensor,
        freqs_cis: torch.Tensor,
        kv_cache: Optional[KVCache] = None,
        layer_idx: int = 0,
        start_pos: int = 0,
        causal: bool = True,
    ) -> torch.Tensor:
        b, s, _ = x.shape

        q = self.wq(x).view(b, s, self.n_heads, self.head_dim)
        k = self.wk(x).view(b, s, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(b, s, self.n_kv_heads, self.head_dim)

        q = apply_rope(q, freqs_cis)
        k = apply_rope(k, freqs_cis)

        q = q.transpose(1, 2)  # (b, n_heads, s, head_dim)
        k = k.transpose(1, 2)  # (b, n_kv_heads, s, head_dim)
        v = v.transpose(1, 2)

        if kv_cache is not None:
            k, v = kv_cache.update(layer_idx, k, v, start_pos)

        k = repeat_kv(k, self.n_rep)  # -> (b, n_heads, total_s, head_dim)
        v = repeat_kv(v, self.n_rep)

        out = F.scaled_dot_product_attention(
            q, k, v,
            # Cached prefill (start_pos == 0, s > 1) must still be causal; single-token
            # decode (s == 1) attends to the whole cache so needs no mask.
            is_causal=causal and s > 1 and start_pos == 0,
            dropout_p=self.dropout if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(b, s, self.n_heads * self.head_dim)
        return self.wo(out)


# ========================================================================
# SwiGLU MLP
# ========================================================================

class SwiGLU(nn.Module):
    def __init__(self, dim: int, mult: float = 4.0):
        super().__init__()
        hidden = int(2 * (dim * mult) / 3)
        hidden = 64 * ((hidden + 63) // 64)  # round up to multiple of 64
        self.w1 = nn.Linear(dim, hidden, bias=False)  # gate
        self.w3 = nn.Linear(dim, hidden, bias=False)  # up
        self.w2 = nn.Linear(hidden, dim, bias=False)  # down

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


# ========================================================================
# Transformer block + full model
# ========================================================================

class Block(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.attn = Attention(cfg)
        self.ffn_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.dim, cfg.ffn_mult)

    def forward(self, x, freqs_cis, kv_cache=None, layer_idx=0, start_pos=0):
        x = x + self.attn(self.attn_norm(x), freqs_cis, kv_cache, layer_idx, start_pos)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm_f = RMSNorm(cfg.dim, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        self._freqs_cache = {}  # (seq_len, scaling) -> freqs_cis, so eval can request longer lengths
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.tok_emb.weight.numel()
        return n

    def get_freqs_cis(self, seq_len: int, device):
        key = seq_len
        if key not in self._freqs_cache:
            self._freqs_cache[key] = precompute_rope_freqs(
                self.cfg.head_dim,
                seq_len,
                theta=self.cfg.rope_theta,
                train_max_seq_len=self.cfg.max_seq_len,
            ).to(device)
        return self._freqs_cache[key]

    def forward(
        self,
        idx: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
        kv_cache: Optional[KVCache] = None,
        start_pos: int = 0,
    ):
        b, s = idx.shape
        device = idx.device
        total_len = start_pos + s
        freqs_cis_full = self.get_freqs_cis(total_len, device)
        freqs_cis = freqs_cis_full[start_pos:start_pos + s]

        x = self.tok_emb(idx)
        for i, block in enumerate(self.blocks):
            x = block(x, freqs_cis, kv_cache=kv_cache, layer_idx=i, start_pos=start_pos)
        x = self.norm_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )
        return logits, loss

    def new_kv_cache(self, batch_size, max_seq_len, device, dtype=torch.float32):
        return KVCache(
            self.cfg.n_layers, batch_size, self.cfg.n_kv_heads, max_seq_len,
            self.cfg.head_dim, device, dtype,
        )


if __name__ == "__main__":
    for variant in ["mha", "gqa8", "gqa4", "mqa"]:
        cfg = make_model_config(variant, vocab_size=512, max_seq_len=1024)
        m = GPT(cfg)
        print(f"{cfg.attention_name():8s} n_kv_heads={cfg.n_kv_heads:2d}  "
              f"params={m.num_params()/1e6:.2f}M  head_dim={cfg.head_dim}")
        x = torch.randint(0, cfg.vocab_size, (2, 32))
        y = torch.randint(0, cfg.vocab_size, (2, 32))
        logits, loss = m(x, y)
        assert logits.shape == (2, 32, cfg.vocab_size)
        print("  forward OK, loss:", loss.item())


In [ ]:
%%writefile data.py
"""
data.py

- TinyStories loading for pre-training (falls back to a synthetic
  TinyStories-style corpus if the `datasets` library / internet isn't
  available, so the pipeline never silently blocks on connectivity).
"""

import os
import random
import re
from typing import List, Tuple

import torch

FALLBACK_SEED_TEXT = """
Once upon a time, in a small village, there lived a little robot named Bolt.
Bolt loved to help his friends. Every morning, Bolt would water the flowers
in the garden. One day, Bolt found a lost kitten near the old oak tree.
The kitten was scared and hungry. Bolt gently picked up the kitten and
carried it home. His friend Mia was very happy to see the kitten safe.
Mia and Bolt decided to name the kitten Whiskers. They gave Whiskers some
milk and a warm blanket. From that day on, Bolt, Mia, and Whiskers were
the best of friends. They played together in the park every afternoon.
One sunny day, a big storm came to the village. The wind was very strong
and the rain fell hard. Bolt was not afraid because he was made of metal.
But Mia and Whiskers were scared, so Bolt built a small shelter for them.
They stayed warm and dry until the storm passed. The next morning, the
sun came out and the village was full of rainbows. Everyone was happy
and thanked Bolt for being so brave and kind. The village threw a party
to celebrate, with cake and games for all the children and their pets.
""".strip()


STORY_SEP = "\n<<<STORY_SEP>>>\n"  # delimiter used only for caching; never appears in real text


def load_tinystories_stories(max_chars: int = 5_000_000, cache_dir: str = "./data_cache") -> List[str]:
    """
    Same loading logic as before, but returns a LIST of individual stories
    instead of one giant concatenated blob. This is what makes a proper
    train/val split possible: HuggingFace's TinyStories is naturally one
    row per story, and we keep that structure all the way through instead
    of flattening it into a single string and splitting by character
    position (which used to cut a story in half wherever the 95% mark
    happened to fall).

    Cache is keyed by size, same as before: if a previous run cached fewer
    characters than this call asks for, it reloads at the larger size.
    """
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, "tinystories.txt")
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            cached_stories = f.read().split(STORY_SEP)
        cached_chars = sum(len(s) for s in cached_stories)
        if cached_chars >= max_chars:
            # trim to budget without splitting a story in half
            out, total = [], 0
            for s in cached_stories:
                if total >= max_chars:
                    break
                out.append(s)
                total += len(s)
            return out
        print(f"[data] cache has {cached_chars} chars, need {max_chars} -- reloading at larger size")

    try:
        from datasets import load_dataset  # type: ignore
        ds = load_dataset("roneneldan/TinyStories", split="train")
        stories = []
        total = 0
        for row in ds:
            t = row["text"]
            stories.append(t)
            total += len(t)
            if total >= max_chars:
                break
        print(f"[data] Loaded real TinyStories: {len(stories)} stories, {total} chars")
    except Exception as e:
        print(f"[data] Could not load TinyStories from HF ({e}). "
              f"Falling back to synthetic TinyStories-style corpus.")
        random.seed(0)
        sentences = re.split(r"(?<=[.!?])\s+", FALLBACK_SEED_TEXT)
        stories = []
        total = 0
        while total < max_chars:
            random.shuffle(sentences)
            story = " ".join(sentences)
            stories.append(story)
            total += len(story)

    with open(cache_path, "w", encoding="utf-8") as f:
        f.write(STORY_SEP.join(stories))
    return stories


def load_tinystories(max_chars: int = 5_000_000, cache_dir: str = "./data_cache") -> str:
    """Back-compat wrapper: returns all loaded stories concatenated into one string."""
    return "\n".join(load_tinystories_stories(max_chars, cache_dir))


def train_val_split_stories(stories: List[str], val_frac: float = 0.05, seed: int = 0) -> Tuple[str, str]:
    """
    Splits by WHOLE STORY, not by character position -- no story is ever
    cut across the train/val boundary.

    The split is content-based (hash of each story's text), not
    position-based: whether a given story lands in train or val depends
    only on the story itself, not on where it sits in the loaded list or
    how many stories were loaded in total. This matters because train.py
    and evaluate.py may load different amounts of data (--max_chars can
    differ), and a position-based shuffle+split would silently produce a
    *different* train/val boundary in each script -- meaning "held-out"
    val data could actually include stories the model trained on. A
    content hash makes the same story always land on the same side,
    regardless of which script loaded it or how much else was loaded
    alongside it.
    """
    import hashlib
    threshold = int(val_frac * 1_000_003)  # large prime-ish modulus for even distribution
    train_stories, val_stories = [], []
    for s in stories:
        h = int(hashlib.md5((str(seed) + s).encode("utf-8")).hexdigest(), 16)
        if (h % 1_000_003) < threshold:
            val_stories.append(s)
        else:
            train_stories.append(s)

    # Safety net for small corpora (e.g. smoke tests): the hash split is
    # probabilistic per-story, so with few stories it can land zero in
    # val by chance. Guarantee at least one story on each side.
    if not val_stories and len(train_stories) > 1:
        val_stories.append(train_stories.pop())
    if not train_stories and val_stories:
        train_stories.append(val_stories.pop())

    return "\n".join(train_stories), "\n".join(val_stories)


def train_val_split(text: str, val_frac: float = 0.05) -> Tuple[str, str]:
    """
    Back-compat wrapper for callers that only have a flat string (no story
    boundaries known). Prefer train_val_split_stories when you have the
    story list -- this one still splits by raw character position.
    """
    n = len(text)
    split = int(n * (1 - val_frac))
    return text[:split], text[split:]


class TokenDataset:
    """Wraps a pre-tokenized 1D LongTensor and serves random fixed-length blocks."""

    def __init__(self, token_ids: torch.Tensor, block_size: int):
        self.ids = token_ids
        self.block_size = block_size

    def __len__(self):
        return max(0, len(self.ids) - self.block_size - 1)

    def get_batch(self, batch_size: int, device: str = "cpu"):
        ix = torch.randint(0, len(self), (batch_size,))
        x = torch.stack([self.ids[i:i + self.block_size] for i in ix])
        y = torch.stack([self.ids[i + 1:i + 1 + self.block_size] for i in ix])
        return x.to(device), y.to(device)


if __name__ == "__main__":
    stories = load_tinystories_stories(max_chars=20_000)
    print(f"loaded {len(stories)} stories, sample: {stories[0][:150].strip()!r}")
    tr, va = train_val_split_stories(stories)
    print("train/val chars:", len(tr), len(va))
    print("last 80 chars of train (should end cleanly, not mid-story-cut):", repr(tr[-80:]))


In [ ]:
%%writefile train.py
"""
train.py

Trains one of {mha, gqa8, gqa4, mqa} ~20M-param models from scratch on
TinyStories with a strict context window of L_train=1024, logging
loss/perplexity to a CSV so Phase 4/5 can consume it without re-running
anything.

Usage:
    python train.py --attention mha  --steps 3000 --out_dir runs/mha
    python train.py --attention gqa8 --steps 3000 --out_dir runs/gqa8
    python train.py --attention gqa4 --steps 3000 --out_dir runs/gqa4
    python train.py --attention mqa  --steps 3000 --out_dir runs/mqa
"""

import argparse
import csv
import math
import os
import time

import torch

from data import load_tinystories_stories, train_val_split_stories, TokenDataset
from model import GPT, make_model_config
from tokenizer import PyTorchBPETokenizer


def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def evaluate_val_loss(model, val_ds, device, block_size, iters=20, batch_size=8):
    model.eval()
    losses = []
    with torch.no_grad():
        for _ in range(iters):
            x, y = val_ds.get_batch(batch_size, device)
            _, loss = model(x, y)
            losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--attention", choices=["mha", "gqa8", "gqa4", "mqa"], required=True)
    ap.add_argument("--vocab_size", type=int, default=2048)
    ap.add_argument("--block_size", type=int, default=1024, help="L_train")
    ap.add_argument("--batch_size", type=int, default=16)
    ap.add_argument("--steps", type=int, default=3000)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--warmup_steps", type=int, default=100)
    ap.add_argument("--eval_every", type=int, default=200)
    ap.add_argument("--out_dir", type=str, required=True)
    ap.add_argument("--max_chars", type=int, default=20_000_000, help="TinyStories chars to load")
    ap.add_argument("--tokenizer_train_chars", type=int, default=2_000_000,
                     help="how much of the (train-split) TinyStories text to run BPE training on")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    os.makedirs(args.out_dir, exist_ok=True)
    device = get_device()
    print(f"[train] device={device} attention={args.attention}")

    # --- Data + tokenizer -------------------------------------------------
    stories = load_tinystories_stories(max_chars=args.max_chars)
    train_text, val_text = train_val_split_stories(stories)
    print(f"[train] {len(stories)} stories loaded -> train {len(train_text)} chars / val {len(val_text)} chars "
          f"(split by whole story, not character position)")

    tok_path = os.path.join("tokenizer_cache")
    if os.path.exists(os.path.join(tok_path, "vocab.json")):
        tokenizer = PyTorchBPETokenizer.load(tok_path)
        print(f"[train] loaded cached tokenizer, vocab_size={tokenizer.vocab_size}")
    else:
        tokenizer = PyTorchBPETokenizer(max_vocab_size=args.vocab_size)
        # Trained on the *train split* of the same TinyStories text the model
        # trains on (not a separate corpus) -- capped at tokenizer_train_chars
        # because BPE training is O(n) per merge in this implementation and
        # doesn't need the full corpus to learn a good merge table.
        tokenizer.train_python(train_text[:args.tokenizer_train_chars], target_vocab_size=args.vocab_size)
        tokenizer.save(tok_path)
        print(f"[train] trained + cached tokenizer on {min(len(train_text), args.tokenizer_train_chars)} chars, "
              f"vocab_size={tokenizer.vocab_size}")

    train_ids = tokenizer.encode_bulk_fast(train_text, device="cpu")
    val_ids = tokenizer.encode_bulk_fast(val_text, device="cpu")
    train_ds = TokenDataset(train_ids, args.block_size)
    val_ds = TokenDataset(val_ids, args.block_size)
    print(f"[train] train tokens={len(train_ids)} val tokens={len(val_ids)}")
    if len(val_ids) < 8192 + 1:
        print(f"[train] WARNING: val split only has {len(val_ids)} tokens -- evaluate.py needs at "
              f"least 8193 to measure perplexity at context_len=8192. Raise --max_chars.")

    # --- Model --------------------------------------------------------------
    cfg = make_model_config(args.attention, vocab_size=tokenizer.vocab_size, max_seq_len=args.block_size)
    model = GPT(cfg).to(device)
    print(f"[train] {cfg.attention_name()} n_kv_heads={cfg.n_kv_heads} params={model.num_params()/1e6:.2f}M")

    optim = torch.optim.AdamW(model.parameters(), lr=args.lr, betas=(0.9, 0.95), weight_decay=0.1)

    def lr_at(step):
        if step < args.warmup_steps:
            return args.lr * step / max(1, args.warmup_steps)
        progress = (step - args.warmup_steps) / max(1, args.steps - args.warmup_steps)
        return 0.5 * args.lr * (1 + math.cos(math.pi * min(1.0, progress)))

    # --- Logging --------------------------------------------------------------
    log_path = os.path.join(args.out_dir, "train_log.csv")
    with open(log_path, "w", newline="") as f:
        csv.writer(f).writerow(["step", "split", "loss", "perplexity", "lr", "wall_time_s"])

    t0 = time.time()
    model.train()
    for step in range(1, args.steps + 1):
        lr = lr_at(step)
        for g in optim.param_groups:
            g["lr"] = lr

        x, y = train_ds.get_batch(args.batch_size, device)
        _, loss = model(x, y)
        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()

        if step % args.eval_every == 0 or step == args.steps:
            val_loss = evaluate_val_loss(model, val_ds, device, args.block_size)
            train_ppl = math.exp(min(20, loss.item()))
            val_ppl = math.exp(min(20, val_loss))
            elapsed = time.time() - t0
            print(f"step {step:5d}/{args.steps} | train_loss {loss.item():.4f} (ppl {train_ppl:.1f}) "
                  f"| val_loss {val_loss:.4f} (ppl {val_ppl:.1f}) | lr {lr:.2e} | {elapsed:.0f}s")
            with open(log_path, "a", newline="") as f:
                w = csv.writer(f)
                w.writerow([step, "train", loss.item(), train_ppl, lr, elapsed])
                w.writerow([step, "val", val_loss, val_ppl, lr, elapsed])

    ckpt_path = os.path.join(args.out_dir, "model.pt")
    torch.save({"model_state": model.state_dict(), "config": cfg}, ckpt_path)
    print(f"[train] saved checkpoint -> {ckpt_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
"""
evaluate.py

Zero-shot long-context evaluation. For each trained checkpoint
(MHA / GQA-8 / GQA-4 / MQA), without any retraining, measures at context
lengths {1024, 2048, 4096, 8192}:

  1. Perplexity vs. context length (held-out TinyStories text) -- primary
     quality metric.
  2. Peak VRAM & memory scaling: torch.cuda.max_memory_allocated over a full
     cached prefill + decode pass, plus the exact KV-cache size in MB.
  3. Inference throughput & latency: prefill tokens/s and ms, and cached
     single-token decode tokens/s and ms/token.

RoPE: the model (see model.py) uses one RoPE variant throughout -- dynamic
NTK-aware scaling, which is mathematically identical to vanilla RoPE up to
L_train and only rescales theta once you extrapolate past it.

Writes one row per (model, context_len) to CSV so visualize.py never has to
touch a checkpoint again.

Usage:
    python evaluate.py --runs_dir runs --out_csv results/eval_results.csv
"""

import argparse
import csv
import gc
import math
import os
import statistics
import time

import torch

from data import load_tinystories_stories, train_val_split_stories
from model import GPT, ModelConfig
from tokenizer import PyTorchBPETokenizer

CONTEXT_LENGTHS = [1024, 2048, 4096, 8192]


def get_device():
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def sync(device):
    if device == "cuda":
        torch.cuda.synchronize()


def load_model(run_dir, device):
    # weights_only=False: checkpoint includes a ModelConfig dataclass, not just
    # tensors. Safe here because these are checkpoints this same pipeline wrote.
    ckpt = torch.load(os.path.join(run_dir, "model.pt"), map_location=device, weights_only=False)
    cfg: ModelConfig = ckpt["config"]
    model = GPT(cfg).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, cfg


@torch.no_grad()
def perplexity_at_length(model, cfg, ids, context_len, device, n_windows=10):
    """Average perplexity over n_windows non-overlapping windows of context_len tokens."""
    losses = []
    max_start = max(1, len(ids) - context_len - 1)
    step = max(1, max_start // n_windows)
    for start in range(0, max_start, step):
        chunk = ids[start:start + context_len + 1]
        if len(chunk) < context_len + 1:
            continue
        x = chunk[:-1].unsqueeze(0).to(device)
        y = chunk[1:].unsqueeze(0).to(device)
        _, loss = model(x, y)
        losses.append(loss.item())
        if len(losses) >= n_windows:
            break
    if not losses:
        return float("nan")
    mean_loss = sum(losses) / len(losses)
    return math.exp(min(20, mean_loss))


@torch.no_grad()
def measure_memory_and_speed(model, cfg, context_len, device, decode_steps=64, n_repeats=3):
    """
    One combined pass per context length:

      prefill : feed `context_len` tokens through the model into a pre-allocated KV cache
      decode  : generate `decode_steps` tokens one at a time from that cache (greedy)

    Reports
      - peak_vram_mb      : torch.cuda.max_memory_allocated over the whole prefill+decode
                            (includes model weights + KV cache + activations). On CPU this
                            is the theoretical KV-cache size instead.
      - kv_cache_mb       : exact KV-cache size (this is where MHA/GQA/MQA differ)
      - prefill_ms / prefill_tok_s
      - decode_ms_per_tok / decode_tok_s   (batch size 1)

    Timing is the median of `n_repeats` runs after one untimed warm-up run.

    RoPE frequencies are precomputed ONCE for the full length and reused for every step.
    Otherwise model.get_freqs_cis would build a new table on CPU at each decode step
    (total_len changes every token), which would pollute both latency and peak VRAM
    with work that has nothing to do with the attention topology.
    """
    max_len = context_len + decode_steps
    x = torch.randint(0, cfg.vocab_size, (1, context_len), device=device)

    freqs_full = model.get_freqs_cis(max_len, device)
    model.get_freqs_cis = lambda seq_len, dev: freqs_full[:seq_len]  # instance override

    def one_pass(cache):
        sync(device)
        t0 = time.perf_counter()
        logits, _ = model(x, kv_cache=cache, start_pos=0)
        nxt = logits[:, -1:].argmax(-1)
        sync(device)
        t1 = time.perf_counter()
        for i in range(decode_steps):
            logits, _ = model(nxt, kv_cache=cache, start_pos=context_len + i)
            nxt = logits[:, -1:].argmax(-1)
        sync(device)
        t2 = time.perf_counter()
        return t1 - t0, t2 - t1

    try:
        # untimed warm-up (cuda kernels, allocator, cudnn autotune)
        cache = model.new_kv_cache(1, max_len, device)
        one_pass(cache)
        del cache
        if device == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device)

        prefill_times, decode_times = [], []
        kv_cache_mb = None
        for _ in range(n_repeats):
            cache = model.new_kv_cache(1, max_len, device)
            kv_cache_mb = cache.nbytes() / (1024 ** 2)
            p, d = one_pass(cache)
            prefill_times.append(p)
            decode_times.append(d)
            del cache

        if device == "cuda":
            peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
            vram_kind = "measured_mb"
        else:
            peak_vram_mb = kv_cache_mb
            vram_kind = "theoretical_kv_mb"
    finally:
        del model.get_freqs_cis  # remove instance override -> class method again
        if device == "cuda":
            torch.cuda.empty_cache()

    prefill_s = statistics.median(prefill_times)
    decode_s = statistics.median(decode_times)
    return {
        "peak_vram_mb": peak_vram_mb,
        "vram_kind": vram_kind,
        "kv_cache_mb": kv_cache_mb,
        "prefill_ms": prefill_s * 1000,
        "prefill_tok_s": context_len / prefill_s,
        "decode_ms_per_tok": decode_s * 1000 / decode_steps,
        "decode_tok_s": decode_steps / decode_s,
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs_dir", type=str, default="runs")
    ap.add_argument("--variants", nargs="+", default=["mha", "gqa8", "gqa4", "mqa"])
    ap.add_argument("--out_csv", type=str, default="results/eval_results.csv")
    ap.add_argument("--decode_steps", type=int, default=64,
                     help="tokens generated one-by-one from the KV cache when timing decode")
    ap.add_argument("--speed_repeats", type=int, default=3,
                     help="timed repeats per (model, context_len); median is reported")
    ap.add_argument("--max_chars", type=int, default=20_000_000,
                     help="must be >= the --max_chars used in train.py, or the val split won't "
                          "have enough tokens for the longer context lengths")
    ap.add_argument("--max_context_for_cpu", type=int, default=2048,
                     help="cap context length on CPU-only machines to keep this runnable")
    args = ap.parse_args()

    device = get_device()
    os.makedirs(os.path.dirname(args.out_csv) or ".", exist_ok=True)

    tokenizer = PyTorchBPETokenizer.load("tokenizer_cache")
    stories = load_tinystories_stories(max_chars=args.max_chars)
    _, val_text = train_val_split_stories(stories)
    val_ids = tokenizer.encode_bulk_fast(val_text, device="cpu")
    print(f"[eval] val tokens={len(val_ids)} (from {len(stories)} stories loaded, "
          f"same content-based split train.py used)")

    desired = CONTEXT_LENGTHS if device == "cuda" else \
        [l for l in CONTEXT_LENGTHS if l <= args.max_context_for_cpu]
    context_lengths = [l for l in desired if l + 1 <= len(val_ids)] or [desired[0]]
    if len(context_lengths) < len(desired):
        skipped = [l for l in desired if l not in context_lengths]
        print(f"[eval] WARNING: val split too small for context lengths {skipped} -- skipping "
              f"those instead of reporting nan. Raise --max_chars to fix.")

    rows = []
    for variant in args.variants:
        run_dir = os.path.join(args.runs_dir, variant)
        if not os.path.exists(os.path.join(run_dir, "model.pt")):
            print(f"[eval] skipping {variant}: no checkpoint at {run_dir}")
            continue
        model, cfg = load_model(run_dir, device)
        print(f"[eval] {variant}: {cfg.attention_name()} n_kv_heads={cfg.n_kv_heads}")

        for ctx_len in context_lengths:
            ppl = perplexity_at_length(model, cfg, val_ids, ctx_len, device)
            m = measure_memory_and_speed(model, cfg, ctx_len, device,
                                          decode_steps=args.decode_steps,
                                          n_repeats=args.speed_repeats)
            print(f"  ctx={ctx_len:5d} ppl={ppl:8.2f} | peak_vram={m['peak_vram_mb']:8.1f}MB "
                  f"kv={m['kv_cache_mb']:7.1f}MB | prefill {m['prefill_tok_s']:9.0f} tok/s "
                  f"| decode {m['decode_tok_s']:7.1f} tok/s ({m['decode_ms_per_tok']:.2f} ms/tok)")
            rows.append({
                "model": variant, "attention": cfg.attention_name(), "n_kv_heads": cfg.n_kv_heads,
                "context_len": ctx_len, "perplexity": ppl, **m,
            })
        del model
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    fieldnames = ["model", "attention", "n_kv_heads", "context_len", "perplexity",
                  "peak_vram_mb", "vram_kind", "kv_cache_mb",
                  "prefill_ms", "prefill_tok_s", "decode_ms_per_tok", "decode_tok_s"]
    with open(args.out_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow(r)
    print(f"[eval] wrote {len(rows)} rows -> {args.out_csv}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile visualize.py
"""
visualize.py

Turns results/eval_results.csv into the figures for the three evaluated metrics:

  1. figures/perplexity_curves.png   - Perplexity vs context length (primary quality metric)
  2. figures/vram_scaling.png        - Peak VRAM and KV-cache size vs context length
  3. figures/throughput_latency.png  - Decode tokens/s, decode ms/token, prefill tokens/s
  4. figures/summary_table.png / .csv - everything side by side, plus % VRAM saved vs MHA

Usage:
    python visualize.py --csv results/eval_results.csv --out_dir figures
"""

import argparse
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")


def _lineplot(ax, df, y, ylabel, title, logy=False):
    for model, g in df.groupby("attention"):
        g = g.sort_values("context_len")
        ax.plot(g["context_len"], g[y], "-", marker="o", label=model)
    ax.set_xscale("log", base=2)
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("Context length (tokens, log scale)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)


def plot_perplexity_curves(df, out_dir):
    sub = df.dropna(subset=["perplexity"])
    fig, ax = plt.subplots(figsize=(9, 6))
    _lineplot(ax, sub, "perplexity", "Perplexity",
              "Zero-shot long-context perplexity\n(trained at L=1024)")
    ax.axvline(1024, color="gray", linestyle=":", linewidth=1)
    ax.text(1024, ax.get_ylim()[1] * 0.95, " L_train", fontsize=10, color="gray")
    ax.legend(fontsize=10, ncol=1, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    path = os.path.join(out_dir, "perplexity_curves.png")
    fig.savefig(path, dpi=200)
    plt.close(fig)
    print(f"[viz] wrote {path}")


def plot_vram_scaling(df, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    _lineplot(axes[0], df, "peak_vram_mb", "Peak VRAM (MB)",
              "Peak VRAM vs context length\n(weights + KV cache + activations)")
    _lineplot(axes[1], df, "kv_cache_mb", "KV cache size (MB)",
              "KV-cache size vs context length")
    axes[1].legend(fontsize=10, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    path = os.path.join(out_dir, "vram_scaling.png")
    fig.savefig(path, dpi=200)
    plt.close(fig)
    print(f"[viz] wrote {path}")


def plot_throughput_latency(df, out_dir):
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))
    _lineplot(axes[0], df, "decode_tok_s", "Tokens / second",
              "Decode throughput (batch 1, cached)")
    _lineplot(axes[1], df, "decode_ms_per_tok", "ms / token",
              "Decode latency per token")
    _lineplot(axes[2], df, "prefill_tok_s", "Tokens / second",
              "Prefill throughput")
    axes[2].legend(fontsize=10, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    path = os.path.join(out_dir, "throughput_latency.png")
    fig.savefig(path, dpi=200)
    plt.close(fig)
    print(f"[viz] wrote {path}")


def build_summary_table(df, out_dir):
    mha = df[df["attention"] == "MHA"].set_index("context_len")
    rows = []
    for _, r in df.sort_values(["context_len", "model"]).iterrows():
        saved_peak = saved_kv = ""
        if r["context_len"] in mha.index:
            base_peak = mha.loc[r["context_len"], "peak_vram_mb"]
            base_kv = mha.loc[r["context_len"], "kv_cache_mb"]
            if base_peak > 0:
                saved_peak = round(100 * (1 - r["peak_vram_mb"] / base_peak), 1)
            if base_kv > 0:
                saved_kv = round(100 * (1 - r["kv_cache_mb"] / base_kv), 1)
        rows.append({
            "model": r["attention"],
            "context_len": int(r["context_len"]),
            "perplexity": round(r["perplexity"], 2),
            "peak_vram_mb": round(r["peak_vram_mb"], 1),
            "kv_cache_mb": round(r["kv_cache_mb"], 1),
            "peak_vram_saved_vs_mha_pct": saved_peak,
            "kv_saved_vs_mha_pct": saved_kv,
            "decode_tok_s": round(r["decode_tok_s"], 1),
            "decode_ms_per_tok": round(r["decode_ms_per_tok"], 2),
            "prefill_tok_s": round(r["prefill_tok_s"], 0),
        })
    table = pd.DataFrame(rows)
    csv_path = os.path.join(out_dir, "summary_table.csv")
    table.to_csv(csv_path, index=False)
    print(f"[viz] wrote {csv_path}")

    fig, ax = plt.subplots(figsize=(18, 0.5 * len(table) + 1.5))
    ax.axis("off")
    tbl = ax.table(cellText=table.values, colLabels=table.columns, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1, 1.6)
    fig.tight_layout()
    png_path = os.path.join(out_dir, "summary_table.png")
    fig.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[viz] wrote {png_path}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", type=str, default="results/eval_results.csv")
    ap.add_argument("--out_dir", type=str, default="figures")
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    df = pd.read_csv(args.csv)

    plot_perplexity_curves(df, args.out_dir)
    plot_vram_scaling(df, args.out_dir)
    plot_throughput_latency(df, args.out_dir)
    build_summary_table(df, args.out_dir)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_all.py
"""
run_all.py

Runs the entire pipeline end to end:
  train mha -> train gqa8 -> train gqa4 -> train mqa -> evaluate -> visualize

This is a thin convenience wrapper; each stage is a real standalone
script (train.py / evaluate.py / visualize.py) that you can also run,
inspect, and re-run independently -- nothing here is hidden behind a
single black-box call.

Usage:
    python run_all.py --steps 3000 --vocab_size 2048
    python run_all.py --steps 200 --vocab_size 512   # quick smoke test
"""

import argparse
import subprocess
import sys


def run(cmd):
    print(f"\n$ {' '.join(cmd)}\n" + "=" * 70)
    subprocess.run(cmd, check=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--steps", type=int, default=3000)
    ap.add_argument("--vocab_size", type=int, default=2048)
    ap.add_argument("--block_size", type=int, default=1024)
    ap.add_argument("--batch_size", type=int, default=16)
    ap.add_argument("--max_chars", type=int, default=20_000_000)
    ap.add_argument("--decode_steps", type=int, default=64)
    ap.add_argument("--variants", nargs="+", default=["mha", "gqa8", "gqa4", "mqa"])
    args = ap.parse_args()

    py = sys.executable

    for variant in args.variants:
        run([
            py, "train.py",
            "--attention", variant,
            "--steps", str(args.steps),
            "--vocab_size", str(args.vocab_size),
            "--block_size", str(args.block_size),
            "--batch_size", str(args.batch_size),
            "--max_chars", str(args.max_chars),
            "--out_dir", f"runs/{variant}",
        ])

    run([
        py, "evaluate.py",
        "--runs_dir", "runs",
        "--variants", *args.variants,
        "--out_csv", "results/eval_results.csv",
        "--decode_steps", str(args.decode_steps),
        "--max_chars", str(args.max_chars),
    ])

    run([py, "visualize.py", "--csv", "results/eval_results.csv", "--out_dir", "figures"])

    print("\nDone. See figures/ for perplexity_curves.png, vram_scaling.png, "
          "throughput_latency.png, summary_table.png/.csv")


if __name__ == "__main__":
    main()


## 3. Quick smoke test (optional, ~1-2 min on GPU)
Verifies the whole pipeline (train x4 -> evaluate -> visualize) before committing to a real run.
Uses tiny settings -- results here are meaningless, this only checks nothing crashes.

In [ ]:
!python run_all.py --steps 50 --vocab_size 512 --max_chars 300000 --decode_steps 8

## 4. Real experiment run
Tokenizing the ~20M-character corpus takes roughly 2 minutes (fast BPE encoder);
`~3000` steps per variant on a T4 should take well under an hour total for a 20M model
at `L=1024`. `--max_chars 20000000` gives both training and the held-out val split enough
tokens to measure perplexity all the way out to `context_len=8192` (a too-small val split
shows up as a warning + skipped context lengths otherwise). Adjust as you like.

In [ ]:
!python run_all.py \
  --steps 3000 \
  --vocab_size 2048 \
  --block_size 1024 \
  --batch_size 32 \
  --max_chars 20000000 \
  --decode_steps 64

## 5. View the figures inline

In [ ]:
from IPython.display import Image, display
import pandas as pd

display(Image('figures/perplexity_curves.png'))
display(Image('figures/vram_scaling.png'))
display(Image('figures/throughput_latency.png'))
display(Image('figures/summary_table.png'))

pd.read_csv('figures/summary_table.csv')

## 6. (Optional) Save results back to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/gqa_research_results
!cp -r figures results runs /content/drive/MyDrive/gqa_research_results/
print('Saved to Google Drive: MyDrive/gqa_research_results')
